# Chapter 6 — RLVR with GRPO on GSM8K (Math Reasoning)

This notebook imports algorithms from the `rlbook` package.
For the zero-dependency verification notebook, see `ch6_rlvr_grpo_colab.ipynb`.

**Pipeline:** Base GPT-2 → SFT (CoT demonstrations) → GRPO (verifiable rewards)


In [ ]:
# Colab setup (uncomment):
# !pip install -q "git+https://github.com/rl-textbook/reinforcement-learning.git#egg=rlbook[alignment]"


In [ ]:
import os, re, math, copy, random, time
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt

from rlbook.algorithms.rlvr import (
    Config, parse_gsm8k_answer, gsm8k_solution_to_cot_and_final,
    MathVerifier, CoTFormatter,
    grpo_step_one_prompt, sync_old_policy,
    pass_at_k, diagnostics_on_test, greedy_response,
    sample_response,
)

SEED = 42
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = Config()
print(f"Device: {device} | Model: {cfg.model_name}")


In [ ]:
# ── Data: GSM8K ───────────────────────────────────────────────────
gsm8k = load_dataset("gsm8k", "main")
train_raw_full = gsm8k["train"].select(range(min(cfg.train_size, len(gsm8k["train"]))))
test_raw = gsm8k["test"].select(range(min(cfg.test_size, len(gsm8k["test"]))))

val_n = min(cfg.val_size, len(train_raw_full))
train_n = max(0, len(train_raw_full) - val_n)
train_raw = train_raw_full.select(range(train_n))
val_raw = train_raw_full.select(range(train_n, train_n + val_n))

def parse_split(raw):
    return [{"question": ex["question"], "answer": parse_gsm8k_answer(ex["answer"]),
             "cot": gsm8k_solution_to_cot_and_final(ex["answer"])} for ex in raw]

train_data, val_data, test_data = parse_split(train_raw), parse_split(val_raw), parse_split(test_raw)
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test (held-out): {len(test_data)}")


In [ ]:
# ── Components ─────────────────────────────────────────────────────
verifier = MathVerifier()
formatter = CoTFormatter()
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
tokenizer.pad_token = tokenizer.eos_token; tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(cfg.model_name).to(device)
base_model.resize_token_embeddings(len(tokenizer))

# Sanity check
r, dbg = verifier.reward("Blue = 18 - 7 = 11. Total = 29.\nFINAL: 29", "29")
print(f"Verifier sanity: reward={r}, {dbg}")


## Base Model Evaluation


In [ ]:
def eval_passks(model, ks, seed_offset=0, label=""):
    passks = {}
    for k in ks:
        seed_everything(SEED + seed_offset + k)
        passks[k] = pass_at_k(model, tokenizer, verifier, formatter, test_data, k,
                               max_new_tokens=cfg.max_new_tokens_eval,
                               temperature=cfg.temperature, top_p=cfg.top_p)
    if label: print(f"{label} " + " ".join(f"Pass@{k}={passks[k]:.3f}" for k in ks))
    return passks

base_pass = eval_passks(base_model, cfg.eval_ks_learning_curves, seed_offset=100, label="Base:")


## SFT Baseline (CoT teacher forcing)


In [ ]:
def make_sft_example(ex):
    prompt = formatter.format(ex["question"])
    full_text = prompt + " " + ex["cot"]
    enc = tokenizer(full_text, truncation=True, max_length=cfg.max_length, return_tensors=None)
    prompt_enc = tokenizer(prompt, truncation=True, max_length=cfg.max_length, return_tensors=None)
    pl = len(prompt_enc["input_ids"])
    labels = enc["input_ids"].copy()
    for i in range(min(pl, len(labels))): labels[i] = -100
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

sft_train = [make_sft_example(ex) for ex in train_data]

def collate_batch(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    def pad(seq, v): return seq + [v]*(max_len - len(seq))
    ids = torch.tensor([pad(x["input_ids"], tokenizer.pad_token_id) for x in batch])
    att = torch.tensor([pad(x["attention_mask"], 0) for x in batch])
    lab = torch.tensor([pad(x["labels"], -100) for x in batch])
    return {"input_ids": ids.to(device), "attention_mask": att.to(device), "labels": lab.to(device)}

train_loader = DataLoader(sft_train, batch_size=cfg.sft_batch_size, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), collate_fn=collate_batch)

sft_model = copy.deepcopy(base_model).to(device)
sft_opt = AdamW(sft_model.parameters(), lr=cfg.sft_lr)

print("=== SFT ===")
it = iter(train_loader)
for step in range(cfg.sft_steps):
    try: batch = next(it)
    except StopIteration: it = iter(train_loader); batch = next(it)
    seed_everything(SEED + step)
    sft_model.train()
    loss = sft_model(**batch).loss
    sft_opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0); sft_opt.step()
    if (step+1) % 50 == 0: print(f"  Step {step+1}/{cfg.sft_steps} | loss={loss.item():.4f}")
print("SFT done.")

sft_pass = eval_passks(sft_model, cfg.eval_ks_learning_curves, seed_offset=200, label="SFT:")


## GRPO (RLVR) Training


In [ ]:
policy = copy.deepcopy(sft_model).to(device)
old_policy = copy.deepcopy(policy).to(device)
grpo_opt = AdamW(policy.parameters(), lr=cfg.grpo_lr)

train_indices = list(range(len(train_data)))
rng = random.Random(SEED); rng.shuffle(train_indices)

# Tracking
eval_x, curve_p1, curve_p4, curve_p8 = [], [], [], []
curve_len, curve_steps, curve_selfcorr = [], [], []
log_loss, log_reward = [], []

def eval_checkpoint(step_idx):
    passks = {}
    for k in cfg.eval_ks_learning_curves:
        seed_everything(SEED + 10_000 + step_idx + k)
        passks[k] = pass_at_k(policy, tokenizer, verifier, formatter, test_data, k,
                               max_new_tokens=cfg.max_new_tokens_eval, temperature=cfg.temperature, top_p=cfg.top_p)
    d = diagnostics_on_test(policy, tokenizer, formatter, test_data, cfg, n_examples=min(64, len(test_data)))
    eval_x.append(step_idx)
    curve_p1.append(passks[1]); curve_p4.append(passks[4]); curve_p8.append(passks[8])
    curve_len.append(d["avg_len_tokens"]); curve_steps.append(d["avg_steps"])
    curve_selfcorr.append(d["self_correction_rate"])
    print(f"  [Eval @ {step_idx}] Pass@1={passks[1]:.3f} Pass@4={passks[4]:.3f} Pass@8={passks[8]:.3f} | "
          f"len={d['avg_len_tokens']:.1f} steps={d['avg_steps']:.1f} selfcorr={d['self_correction_rate']:.2f}")

early_policy = copy.deepcopy(policy).to(device)
eval_checkpoint(0)

t0 = time.time()
for step in range(1, cfg.grpo_steps + 1):
    ex = train_data[train_indices[(step-1) % len(train_indices)]]
    seed_everything(SEED + 50_000 + step)
    loss, metrics = grpo_step_one_prompt(
        policy, old_policy, tokenizer, verifier, formatter,
        question_text=ex["question"], answer_text=ex["answer"],
        group_size=cfg.group_size, clip_eps=cfg.clip_eps, adv_eps=cfg.adv_eps)
    grpo_opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); grpo_opt.step()
    if step % cfg.sync_old_every == 0: sync_old_policy(old_policy, policy)
    log_loss.append(loss.item()); log_reward.append(metrics["mean_reward"])
    if step % 10 == 0:
        print(f"  GRPO step {step:4d} | loss={log_loss[-1]:.4f} | reward={log_reward[-1]:.3f} ± {metrics['reward_std']:.3f}")
    if step % cfg.eval_every == 0: eval_checkpoint(step)
print(f"GRPO done in {(time.time()-t0)/60:.1f} min")


## Results


In [ ]:
# Final held-out comparison
final_base = eval_passks(base_model, cfg.eval_ks_learning_curves, seed_offset=111, label="Base:")
final_sft = eval_passks(sft_model, cfg.eval_ks_learning_curves, seed_offset=222, label="SFT:")
final_grpo = eval_passks(policy, cfg.eval_ks_learning_curves, seed_offset=333, label="GRPO:")

diag_b = diagnostics_on_test(base_model, tokenizer, formatter, test_data, cfg, n_examples=min(64, len(test_data)))
diag_s = diagnostics_on_test(sft_model, tokenizer, formatter, test_data, cfg, n_examples=min(64, len(test_data)))
diag_g = diagnostics_on_test(policy, tokenizer, formatter, test_data, cfg, n_examples=min(64, len(test_data)))

print("\n" + "="*70)
print(f"{'Model':12s}  {'Pass@1':>7s} {'Pass@4':>7s} {'Pass@8':>7s}  {'Len':>6s} {'Steps':>6s} {'SelfCorr':>8s}")
for lbl, pk, d in [("Base", final_base, diag_b), ("SFT", final_sft, diag_s), ("GRPO", final_grpo, diag_g)]:
    print(f"{lbl:12s}  {pk[1]:7.3f} {pk[4]:7.3f} {pk[8]:7.3f}  {d['avg_len_tokens']:6.1f} {d['avg_steps']:6.1f} {d['self_correction_rate']:8.3f}")


## Learning Curves


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(eval_x, curve_p1, label="Pass@1"); axes[0].plot(eval_x, curve_p4, label="Pass@4")
axes[0].plot(eval_x, curve_p8, label="Pass@8"); axes[0].axhline(final_sft[1], ls="--", label="SFT Pass@1")
axes[0].set_xlabel("GRPO steps"); axes[0].set_ylabel("Pass@k (held-out)")
axes[0].set_title("Learning Curves"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(eval_x, curve_len, label="Avg tokens"); axes[1].set_xlabel("GRPO steps")
axes[1].set_ylabel("Response length"); axes[1].set_title("Response Length"); axes[1].legend(); axes[1].grid(True)

axes[2].plot(eval_x, curve_selfcorr, label="Self-correction rate")
axes[2].set_xlabel("GRPO steps"); axes[2].set_ylabel("Rate")
axes[2].set_title("Self-Correction"); axes[2].legend(); axes[2].grid(True)

plt.tight_layout(); plt.show()


## Test-Time Compute Scaling


In [ ]:
measured = {}
for k in cfg.eval_ks_compute_scaling:
    seed_everything(SEED + 7000 + k)
    measured[k] = pass_at_k(policy, tokenizer, verifier, formatter, test_data, k,
                             max_new_tokens=cfg.max_new_tokens_eval, temperature=cfg.temperature, top_p=cfg.top_p)

p = measured[1]
theory = {k: 1.0 - (1.0 - p)**k for k in cfg.eval_ks_compute_scaling}
ks = list(cfg.eval_ks_compute_scaling)

plt.figure(figsize=(8,5))
plt.plot(ks, [measured[k] for k in ks], "o-", label="Measured Pass@N")
plt.plot(ks, [theory[k] for k in ks], "o--", label="Independence theory")
plt.grid(True); plt.xlabel("N samples"); plt.ylabel("Success rate (held-out)")
plt.title("Test-Time Compute Scaling"); plt.legend(); plt.show()
print("Measured:", {k: f"{v:.3f}" for k,v in measured.items()})


## Qualitative Example


In [ ]:
ex = test_data[0]
prompt = formatter.format(ex["question"])
print(f"Problem: {ex['question'][:200]}")
print(f"Ground truth: {ex['answer']}\n")
for lbl, m in [("Early (pre-GRPO)", early_policy), ("GRPO (post-training)", policy)]:
    seed_everything(SEED + 1234)
    resp = greedy_response(m, tokenizer, prompt, max_new_tokens=128)
    print(f"--- {lbl} ---\n{resp[:400]}\n")
